# CCE PoC v3: multi-model + semantic-entropy ablation

Runs the full v3 PoC on **all three LLMs in a single notebook execution**, then aggregates. Each model is launched as a separate subprocess so VRAM is freed cleanly between runs.

**What v3 adds over v2:**
- 3 models (CodeLlama-7B, Qwen2.5-Coder-7B, DeepSeek-Coder-7B)
- Semantic entropy (Kuhn 2023 / Farquhar 2024) as a 7th feature group
- 3 new ablation arms: `semantic_entropy_only`, `flare_plus_se`, `all_features`

**Total wall-clock on A100:** ~4.5 hours (3 models × 90 min each).

**Resumable:** if Colab disconnects mid-run, just open the notebook again and run all cells. Each model checks for cached features/SE/phase3 on Drive and skips what's done.

In [ ]:
# All three models, run sequentially in this notebook.
# Each is a separate subprocess (`!python ...`) so VRAM is freed between runs.
# Total compute on A100: ~4.5 hours. On disconnect, just re-run — Drive cache resumes.

MODELS = [
    "Qwen/Qwen2.5-Coder-7B-Instruct",                # run 1: not gated, smallest risk
    "deepseek-ai/deepseek-coder-7b-instruct-v1.5",   # run 2: not gated
    "codellama/CodeLlama-7b-Instruct-hf",            # run 3: gated; license already accepted from v2
]
print(f'will run {len(MODELS)} models sequentially')

## 2. Install + clone

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers scikit-learn

In [ ]:
import os
if not os.path.exists('/content/reposynth'):
    !git clone https://github.com/aniJani/reposynth.git /content/reposynth
%cd /content/reposynth
!git checkout Research
!git pull --rebase || true

## 3. HuggingFace login (CodeLlama and DeepSeek are gated; Qwen is not)

In [ ]:
from huggingface_hub import login
from google.colab import userdata
try:
    login(userdata.get('HF_TOKEN'))
except Exception as e:
    print('Set HF_TOKEN as a Colab secret first:', e)

## 4. Mount Drive (per-model caches survive disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/cce_poc_v3'
!mkdir -p {OUT_DIR}
print(f'will write per-model results to {OUT_DIR}')

## 5. Run all three models sequentially

Each model is a separate subprocess — clean VRAM, isolated failures. If one model fails, the others still run; you can rerun this cell to retry.

In [ ]:
import time, subprocess

%cd /content/reposynth

for i, model in enumerate(MODELS, 1):
    print(f"\n{'='*70}\n[{i}/{len(MODELS)}] running {model}\n{'='*70}", flush=True)
    t0 = time.time()
    rc = subprocess.call([
        "python", "/content/reposynth/research/paper/cce_poc_v3.py",
        "--model", model,
        "--repos-dir", "/content",
        "--out-dir", OUT_DIR,
    ])
    dt = (time.time() - t0) / 60
    if rc == 0:
        print(f"\n[{i}/{len(MODELS)}] ✓ {model}  ({dt:.1f} min)", flush=True)
    else:
        print(f"\n[{i}/{len(MODELS)}] ✗ {model}  rc={rc}  ({dt:.1f} min)", flush=True)
        print("Continuing to next model. Re-run this cell later to retry failures.", flush=True)
print("\nAll models attempted. Inspect individual results below or skip to aggregation.")

## 6. Inspect per-model results

In [ ]:
import json, re, glob

for model in MODELS:
    slug = re.sub(r'[^A-Za-z0-9]+', '_', model).strip('_')
    path = f'{OUT_DIR}/results__{slug}.json'
    try:
        with open(path) as f:
            res = json.load(f)
    except FileNotFoundError:
        print(f"\n=== {model} ===  (results not found at {path})")
        continue
    print(f"\n=== {res['model']}  (n_tasks={res['n_tasks']}) ===")
    print(f"{'arm':<22} {'#feat':>5} {'acc':>5} {'prec':>5} {'rec':>5} {'f1':>5}")
    for arm, r in res['phase2_loo_ablation'].items():
        print(f"{arm:<22} {r['n_features']:>5d} {r['accuracy']:>5.3f} "
              f"{r['precision']:>5.3f} {r['recall']:>5.3f} {r['f1']:>5.3f}")
    if res.get('phase3_end_to_end'):
        print('Phase 3:')
        print(f"  {'arm':<22} {'final':>5} {'always':>5} {'used':>5} {'save%':>6}")
        for arm, r in res['phase3_end_to_end'].items():
            print(f"  {arm:<22} {r['final_accuracy']:>5.3f} {r['always_retrieve_accuracy']:>5.3f} "
                  f"{r['n_retrievals_used']:>5d} {r['retrieval_save_rate']*100:>5.1f}%")

---
## 7. Aggregation across models *(run this cell ONCE after all three model runs complete)*

In [ ]:
%cd /content/reposynth
!python /content/reposynth/research/paper/cce_poc_v3_aggregate.py \
    --in-dir {OUT_DIR} \
    --out    {OUT_DIR}/v3_combined_results.json